In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
file_path = "../data/processed/customer_features_raw.csv"

customer_df = pd.read_csv(file_path)

print("Shape:", customer_df.shape)
print("\nColumns:")
print(customer_df.columns.tolist())

display(customer_df.head())

Shape: (5350, 7)

Columns:
['Customer ID', 'Recency', 'Frequency', 'Monetary', 'Total_Quantity', 'Unique_Products', 'Average_Order_Value']


,Customer ID,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
0,"12,346.00",326,12,"77,556.46",74285,27,"6,463.04"
1,"12,608.00",405,1,415.79,323,16,415.79
2,"12,745.00",487,2,723.85,467,20,361.93
3,"12,746.00",541,1,254.55,97,17,254.55
4,"12,747.00",2,26,"8,898.48",2640,85,342.25


In [3]:
features = [
    "Recency",
    "Frequency",
    "Monetary",
    "Total_Quantity",
    "Unique_Products",
    "Average_Order_Value",
]

X = customer_df[features].copy()

print("Feature matrix shape:", X.shape)
print("\nFeatures:")
print(X.columns.tolist())

display(X.head())

Feature matrix shape: (5350, 6)

Features:
['Recency', 'Frequency', 'Monetary', 'Total_Quantity', 'Unique_Products', 'Average_Order_Value']


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
0,326,12,"77,556.46",74285,27,"6,463.04"
1,405,1,415.79,323,16,415.79
2,487,2,723.85,467,20,361.93
3,541,1,254.55,97,17,254.55
4,2,26,"8,898.48",2640,85,342.25


In [4]:
skewness = X.skew().sort_values(ascending=False)

print("===== FEATURE SKEWNESS =====")
display(skewness.to_frame("Skewness"))

===== FEATURE SKEWNESS =====


,Skewness
Average_Order_Value,58.18
Monetary,28.19
Total_Quantity,16.31
Frequency,10.43
Unique_Products,5.17
Recency,0.87


In [5]:
log_features = [
    "Frequency",
    "Monetary",
    "Total_Quantity",
    "Unique_Products",
    "Average_Order_Value",
]

X_log = X.copy()

for col in log_features:
    X_log[col] = np.log1p(X_log[col])

print("Log transformation applied to:")
print(log_features)

display(X_log.head())

Log transformation applied to:
['Frequency', 'Monetary', 'Total_Quantity', 'Unique_Products', 'Average_Order_Value']


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
0,326,2.56,11.26,11.22,3.33,8.77
1,405,0.69,6.03,5.78,2.83,6.03
2,487,1.10,6.59,6.15,3.04,5.89
3,541,0.69,5.54,4.58,2.89,5.54
4,2,3.30,9.09,7.88,4.45,5.84


In [6]:
skewness_after = X_log.skew().sort_values(ascending=False)

print("===== SKEWNESS AFTER LOG TRANSFORMATION =====")
display(skewness_after.to_frame("Skewness"))

===== SKEWNESS AFTER LOG TRANSFORMATION =====


,Skewness
Frequency,0.97
Recency,0.87
Monetary,0.25
Average_Order_Value,-0.07
Total_Quantity,-0.09
Unique_Products,-0.28


In [7]:
print("===== OUTLIERS AFTER LOG TRANSFORMATION =====")

for col in features:
    Q1 = X_log[col].quantile(0.25)
    Q3 = X_log[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((X_log[col] < lower) | (X_log[col] > upper)).sum()

    print(f"{col}: {outliers:,} outliers")

===== OUTLIERS AFTER LOG TRANSFORMATION =====
Recency: 0 outliers
Frequency: 28 outliers
Monetary: 54 outliers
Total_Quantity: 85 outliers
Unique_Products: 3 outliers
Average_Order_Value: 147 outliers


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_log)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=features,
    index=X_log.index
)

print("Scaled feature matrix shape:", X_scaled.shape)

display(X_scaled.head())

Scaled feature matrix shape: (5350, 6)


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
0,0.59,1.25,3.26,3.37,-0.34,4.53
1,0.96,-1.06,-0.54,-0.25,-0.75,0.65
2,1.35,-0.56,-0.14,-0.01,-0.58,0.46
3,1.61,-1.06,-0.90,-1.05,-0.70,-0.04
4,-0.96,2.15,1.69,1.15,0.57,0.38


In [9]:
print("===== SCALED FEATURE SUMMARY =====")
display(X_scaled.describe().T)

===== SCALED FEATURE SUMMARY =====


,count,mean,std,min,25%,50%,75%,max
Recency,"5,350.00",0.00,1.00,-0.96,-0.84,-0.50,0.85,2.55
Frequency,"5,350.00",0.00,1.00,-1.06,-1.06,-0.21,0.65,5.26
Monetary,"5,350.00",-0.00,1.00,-3.93,-0.71,-0.04,0.66,4.73
Total_Quantity,"5,350.00",-0.00,1.00,-3.64,-0.64,-0.01,0.67,3.99
Unique_Products,"5,350.00",0.00,1.00,-2.50,-0.66,0.05,0.73,3.25
Average_Order_Value,"5,350.00",-0.00,1.00,-5.93,-0.59,0.04,0.57,8.16


In [10]:
correlation_matrix = X_log.corr()

print("===== FEATURE CORRELATION MATRIX =====")
display(correlation_matrix.round(2))

===== FEATURE CORRELATION MATRIX =====


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
Recency,1.00,-0.51,-0.48,-0.49,-0.43,-0.18
Frequency,-0.51,1.00,0.86,0.80,0.69,0.22
Monetary,-0.48,0.86,1.00,0.93,0.73,0.68
Total_Quantity,-0.49,0.80,0.93,1.00,0.74,0.62
Unique_Products,-0.43,0.69,0.73,0.74,1.00,0.41
Average_Order_Value,-0.18,0.22,0.68,0.62,0.41,1.00


In [11]:
X_final = X_scaled.copy()

print("===== FINAL FEATURE MATRIX =====")
print("Shape:", X_final.shape)
print("\nFeatures:")
print(X_final.columns.tolist())

print("\nMissing values:")
print(X_final.isna().sum())

display(X_final.head())

===== FINAL FEATURE MATRIX =====
Shape: (5350, 6)

Features:
['Recency', 'Frequency', 'Monetary', 'Total_Quantity', 'Unique_Products', 'Average_Order_Value']

Missing values:
Recency                0
Frequency              0
Monetary               0
Total_Quantity         0
Unique_Products        0
Average_Order_Value    0
dtype: int64


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
0,0.59,1.25,3.26,3.37,-0.34,4.53
1,0.96,-1.06,-0.54,-0.25,-0.75,0.65
2,1.35,-0.56,-0.14,-0.01,-0.58,0.46
3,1.61,-1.06,-0.90,-1.05,-0.70,-0.04
4,-0.96,2.15,1.69,1.15,0.57,0.38


In [12]:
from pathlib import Path

output_path = Path("../data/processed/customer_features_engineered.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

X_final.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {X_final.shape}")

Saved: ../data/processed/customer_features_engineered.csv
Shape: (5350, 6)


# Customer Feature Engineering

## 1. Objective

The objective of this notebook is to prepare customer-level features for clustering.

The raw customer features contain several highly right-skewed variables, where a small number of customers have extremely large values. These distributions can negatively influence distance-based clustering algorithms.

The preprocessing steps performed in this notebook are:

1. Load the customer-level dataset.
2. Select the clustering features.
3. Analyze feature skewness.
4. Apply logarithmic transformation to strongly skewed features.
5. Re-check skewness after transformation.
6. Analyze remaining outliers.
7. Standardize all features.
8. Analyze feature correlations.
9. Create the final engineered feature matrix.
10. Save the engineered features for clustering experiments.

---

## 2. Input Dataset

The customer-level dataset contains **5,350 customers** and the following features:

* Recency
* Frequency
* Monetary
* Total Quantity
* Unique Products
* Average Order Value

`Customer ID` is excluded from the clustering feature matrix because it is an identifier rather than a behavioral feature.

---

## 3. Initial Feature Skewness

The original customer features were highly right-skewed:

| Feature             | Skewness |
| ------------------- | -------: |
| Average Order Value |    58.18 |
| Monetary            |    28.19 |
| Total Quantity      |    16.31 |
| Frequency           |    10.43 |
| Unique Products     |     5.17 |
| Recency             |     0.87 |

The strong positive skew indicates that a relatively small number of customers have extremely high values.

---

## 4. Log Transformation

A `log1p` transformation was applied to the strongly skewed positive features:

* Frequency
* Monetary
* Total Quantity
* Unique Products
* Average Order Value

Recency was not transformed because its skewness was comparatively low.

After transformation, skewness was substantially reduced:

| Feature             | Skewness After Transformation |
| ------------------- | ----------------------------: |
| Frequency           |                          0.97 |
| Recency             |                          0.87 |
| Monetary            |                          0.25 |
| Average Order Value |                         -0.07 |
| Total Quantity      |                         -0.09 |
| Unique Products     |                         -0.28 |

The logarithmic transformation successfully compressed the extreme right tails.

---

## 5. Outlier Analysis

Outliers were checked using the IQR method after log transformation.

The remaining outliers were:

| Feature             | Outliers |
| ------------------- | -------: |
| Recency             |        0 |
| Frequency           |       28 |
| Monetary            |       54 |
| Total Quantity      |       85 |
| Unique Products     |        3 |
| Average Order Value |      147 |

These observations were **not removed**.

The remaining extreme customers may represent genuine high-value or high-activity customers. Removing them could eliminate meaningful customer segments.

The log transformation was therefore preferred over deleting these customers.

---

## 6. Feature Scaling

The transformed features were standardized using `StandardScaler`.

This gives the clustering algorithms features on a comparable scale, preventing variables with larger numerical ranges from dominating distance calculations.

The resulting feature matrix contains:

* **5,350 customers**
* **6 features**
* Approximately zero mean for every feature
* Unit standard deviation for every feature

---

## 7. Feature Correlation

Correlation analysis was performed on the log-transformed features.

Several strong relationships were observed, particularly between:

* Frequency and Monetary
* Monetary and Total Quantity
* Frequency and Total Quantity
* Monetary and Unique Products

These correlations are expected because highly active customers generally place more orders, purchase more products, and spend more money.

The features were **not removed at this stage** because each represents a different aspect of customer behavior.

Clustering experiments will determine whether the feature set produces useful and well-separated customer segments.

---

## 8. Final Feature Matrix

The final engineered matrix contains:

```text
Shape: (5350, 6)
```

Features:

```text
Recency
Frequency
Monetary
Total_Quantity
Unique_Products
Average_Order_Value
```

Validation confirmed that there are no missing values in the final matrix.

The final features are:

1. Log-transformed where appropriate
2. Standardized
3. Ready for clustering experiments

---

## 9. Output

The final engineered feature matrix was saved to:

```text
data/processed/customer_features_engineered.csv
```

This file will be used as the input for the clustering experiments in the next notebook.

---

## 10. Conclusion

The customer-level data has now been transformed into a clean clustering-ready feature matrix.

The preprocessing pipeline reduced extreme skewness, preserved legitimate customer outliers, standardized the feature scales, and retained the six behavioral dimensions needed for segmentation.

The next stage is to evaluate different clustering algorithms and determine an appropriate number of customer segments.
